In [1]:
from totalsegmentator.python_api import totalsegmentator
from pathlib import Path
import yaml
import time
import json

In [2]:
def get_organ_task (organ_yaml: str):
    with open(organ_yaml,"r") as f:
        organs = yaml.safe_load(f)
        
    organs_in_total = []
    organs_in_head = []
    all_organs = []
    tasks = set()
    
    for organ in organs["organs"]:
        all_organs.append(organ["source_class"]) #get all required organ 
        tasks.add(organ["task"])
        
        if (organ["task"]) == "total":
            organs_in_total.append(organ["source_class"]) #get organ in total (for roi_subset)
        else:
            organs_in_head.append(organ["source_class"]) #get organ in head_glands_cavities (for masks selection)
            
    return all_organs, organs_in_total, organs_in_head, tasks   
    #list of all organ, organ in total (for roi_subset), organ in head_glands_cavities (for masks selection)

In [ ]:
# data path = path to parent data folder, organ_yaml = path to organ list
def get_params (data_path: str, organ_yaml: str):
    data_folder = Path(data_path)
    input_folder = data_folder / "input"
    output_folder = data_folder / "transit_folder" 
    params = []
    _, roi_subset, _, tasks = get_organ_task(organ_yaml) # get tasks and roi_subset
    
    for subject_dir in sorted(input_folder.iterdir()): #get subjects in input folder Subject = quadra 1-48
        if subject_dir.is_dir():
            subject_img_dirs = sorted(subject_dir.glob("*.nii.gz"))
            
            if not subject_img_dirs:
                raise ValueError (f"WARNING!!! images not found in this dir: '{subject_dir}'") 
            else:

                #get the output dir for TotalSegmentator
                for subject_img_dir in subject_img_dirs: 
                    session_name = subject_img_dir.name.removesuffix("_CT-AC.nii.gz") # get session_name --> Session = test, retest
                    subject_mask_dir = output_folder / subject_dir.name / session_name # formulating output dirs for TotalSegmentator
                    subject_mask_dir.mkdir(parents=True, exist_ok=True) #creating folder based on formulated dirs
                    
                    #listing the parameter
                    for task in tasks:
                        param = {
                            "subject_id": subject_dir.name,
                            "session": session_name,
                            "task": task,
                            "input": subject_img_dir,
                            "output": subject_mask_dir,
                            "device": "gpu"
                        }
                        if task == "total":
                            param["roi_subset"] = roi_subset #add roi_subset if the task is "toal"
                        params.append(param)
    
    
    return params #list of dict{parameter} (subject, session, task, input dir, output dir, device)



In [4]:
parameters

[{'subject_id': 'QUADRA_HC_021',
  'session': 'retest',
  'task': 'total',
  'input': PosixPath('/workspaces/segmentation/data/input/QUADRA_HC_021/retest_CT-AC.nii.gz'),
  'output': PosixPath('/workspaces/segmentation/data/transit_folder/QUADRA_HC_021/retest'),
  'device': 'gpu',
  'roi_subset': ['brain',
   'skull',
   'vertebrae_C1',
   'vertebrae_C4',
   'vertebrae_C7',
   'lung_upper_lobe_left',
   'lung_lower_lobe_left',
   'lung_upper_lobe_right',
   'lung_middle_lobe_right',
   'lung_lower_lobe_right',
   'heart',
   'esophagus',
   'vertebrae_T4',
   'vertebrae_T8',
   'vertebrae_T12',
   'rib_left_6',
   'rib_right_6',
   'liver',
   'kidney_left',
   'kidney_right',
   'pancreas',
   'duodenum',
   'small_bowel',
   'colon',
   'vertebrae_L1',
   'vertebrae_L3',
   'vertebrae_L5',
   'urinary_bladder',
   'prostate',
   'sacrum',
   'hip_left',
   'hip_right']},
 {'subject_id': 'QUADRA_HC_021',
  'session': 'retest',
  'task': 'head_glands_cavities',
  'input': PosixPath('/wo

In [12]:
#writing json report
def write_report_json (parameters):
    json_dir = "/workspaces/segmentation/data/output/segmentation_parameters.json"
    json_path = Path(json_dir)
    
    #write the current parameters
    parameter_json = {
        "subject": parameters["subject_id"],
        "session": parameters["session"],
        "input": str(parameters["input"]),
        "output": str(parameters["output"]),
        "task": parameters["task"],
        "roi_subset": parameters.get("roi_subset", None),
        "device": parameters["device"]
    }

    #get the written parameters
    if file_path.exist() and file.path.stat().st_size > 0 :
        with open (json_path, "r", encoding="utf-8") as file:
            try:
                current_data = json.load(file)
                if not isinstance(current_data, list):
                    current_data = []
            except json.JSONDecodeError:
                current_data = []
    else:
        current_data = []

    # add the current parameters to the written parameters (to update the parameters)
    current_data.append(parameter_json)

    with open(json_path, "w", encoding="utf-8") as file:
        json.dump(current_data, file, indent=4)

In [13]:
# main function to run the segmentation
def run_segmentation (data_path: str, organ_yaml: str):
    parameters = get_params (data_path, organ_yaml)
    start_time = time.perf_counter()
    
    for parameter in parameters:

        #run the segmentation
        if __name__ == "__main__":
            totalsegmentator(
                input = parameter["input"],
                output = parameter["output"],
                task = parameter["task"],
                roi_subset = parameter.get("roi_subset", None),
                device = parameter["device"]
            )

        write_report_json(parameters) # write the json file after one segmentation run

    end_time = time.perf_counter()
    execution_time = (end_time - start_time)/60 # calculating the running time
    print(f"Segmentation time: {execution_time:.6f} minutes")
    
    


In [14]:
run_segmentation("/workspaces/segmentation/data", "/workspaces/segmentation/organs.yaml")

No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024



KeyboardInterrupt: 

In [ ]:
# this is just simulation to see how the parameter looks like in the main function

start_time = time.perf_counter()
for parameter in parameters:
    print (
        "input = ", parameter["input"], "\n",
        "output = ", parameter["output"], "\n",
        "task = ", parameter["task"], "\n",
        "device = ", parameter["device"], "\n"
        "roi_subset = ", parameter.get("roi_subset", None)
    )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


these two cells bellow are for testing time difference between using roi_subset or not

In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            roi_subset = parameter.get("roi_subset", None),
            device = parameter["device"]
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


In [ ]:
start_time = time.perf_counter()
for parameter in parameters:
    if __name__ == "__main__":
        totalsegmentator(
            input = parameter["input"],
            output = parameter["output"],
            task = parameter["task"],
            device = parameter["device"],
        )
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Segmentation time: {execution_time:.6f} seconds")


Segmentation time: 463.452297 seconds
Segmentation time: 1521.024349 seconds